# 🧠 Unsupervised Learning - Assignment: Session 3

**Topic:** Density-Based Spatial Clustering (DBSCAN), Outlier & Noise Detection, Parameter Sensitivity (`eps` & `min_samples`), DBSCAN vs. K-Means Comparison, and Non-Spherical Manifold Clustering

---

### 📋 Assignment Roadmap
1. **Task 1:** Load Mall Customers Dataset, Apply DBSCAN on Income vs. Spending, and Report Cluster & Noise Counts
2. **Task 2:** Publication-Quality Matplotlib Visualization: Differentiated Cluster Hues with Noise Points Marked in Black
3. **Task 3:** Hyperparameter Sensitivity Analysis (`eps` and `min_samples` Exploration & Trade-off Summary)
4. **Task 4:** Comparative Study: DBSCAN vs. K-Means Side-by-Side on Cluster Geometry and Outlier Handling
5. **Task 5:** Non-Spherical Manifold Clustering (`make_moons`) & Extraction of Noise Point Indices

---

## 1️⃣ Task 1: DBSCAN Clustering on Mall Customer Dataset

### 📖 Conceptual Definition & Foundations of DBSCAN
**DBSCAN** stands for **Density-Based Spatial Clustering of Applications with Noise**. Unlike centroid-based algorithms (such as K-Means) that partition space into convex Voronoi cells, DBSCAN groups points based on spatial density connectivity.

#### Core Mathematical Concepts:
1. **Epsilon Neighborhood ($\epsilon$ or `eps`):** The radius around any point $p$ defining its local neighborhood:
   $$\mathcal{N}_{\epsilon}(p) = \{q \in D \mid \text{dist}(p, q) \le \epsilon\}$$
2. **Minimum Points (`min_samples`):** The threshold count of points required within $\mathcal{N}_{\epsilon}(p)$ for $p$ to be considered a dense region.
3. **Classification of Points:**
   - **Core Point:** $|\mathcal{N}_{\epsilon}(p)| \ge \text{min\_samples}$ (dense nucleus of a cluster).
   - **Border Point:** Within distance $\epsilon$ of a Core Point, but has fewer than `min_samples` points in its own neighborhood.
   - **Noise / Outlier:** Neither a Core Point nor reachable from any Core Point. DBSCAN assigns noise points the special cluster label **`-1`**.

---

### 🎯 Target Features for Customer Segmentation
- **Feature 1:** `Annual Income (k$)` — Customer's yearly earnings.
- **Feature 2:** `Spending Score (1-100)` — Behavioral spending metric assigned by the mall.

*Note on Feature Scaling:* Because Euclidean distance is used in neighborhood calculations, features must be standardized using `StandardScaler` so that neither feature dominates the $\epsilon$-ball radius.

In [ ]:
# =====================================================================
# TASK 1: Load Mall Customers & Apply DBSCAN Clustering
# =====================================================================
import os
import pandas as pd
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler

# 1. Load the Mall Customers dataset (local fallback or direct GitHub raw URL)
csv_path = 'Mall_Customers.csv'
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    url = 'https://raw.githubusercontent.com/MainakRepositor/Datasets/master/Mall_Customers.csv'
    df = pd.read_csv(url)

# Normalize column names to avoid whitespace inconsistencies
df.columns = df.columns.str.strip()

# 2. Extract the target clustering features
features = ['Annual Income (k$)', 'Spending Score (1-100)']
X = df[features].values

# 3. Standardize features to equalize scales (mean=0, variance=1)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Apply DBSCAN with calibrated parameters (eps=0.35, min_samples=5)
dbscan = DBSCAN(eps=0.35, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_scaled)

# Store cluster assignments in DataFrame
df['DBSCAN_Cluster'] = dbscan_labels

# 5. Calculate cluster count and noise count
unique_labels = set(dbscan_labels)
n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print("=" * 65)
print("TASK 1: DBSCAN CLUSTERING RESULTS SUMMARY")
print("=" * 65)
print(f"Total Customer Records Analyzed : {len(df)}")
print(f"Selected Features               : {features}")
print(f"DBSCAN Hyperparameters          : eps=0.35, min_samples=5")
print(f"Number of Clusters Discovered   : {n_clusters}")
print(f"Number of Noise Points Detected : {n_noise} ({(n_noise / len(df)) * 100:.1f}% of total data)")
print(f"Unique Cluster Labels Assigned  : {sorted(list(unique_labels))}")
print("=" * 65)

# Preview sample clustered rows
print("\nSample Data Preview with DBSCAN Labels:")
print(df[['CustomerID', 'Annual Income (k$)', 'Spending Score (1-100)', 'DBSCAN_Cluster']].head(10).to_string(index=False))

## 2️⃣ Task 2: Visualizing DBSCAN Results (Clusters & Noise in Black)

### 🎨 Visualization Guidelines
- **Clusters:** Plotted in distinct, vibrant hues to highlight coherent customer cohorts.
- **Noise Points (`label == -1`):** Highlighted prominently in **solid black (`#000000`)** using an **'X'** marker to make outliers immediately identifiable.
- **Domain Context:** We plot on the original feature scale (Annual Income in $k$ vs. Spending Score 1-100) for clear business interpretation.

In [ ]:
# =====================================================================
# TASK 2: Matplotlib Visualization of DBSCAN Clusters and Noise Points
# =====================================================================
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

# Curated aesthetic palette for regular clusters
palette = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12', '#1abc9c', '#d35400']

# 1. Plot regular dense clusters
cluster_ids = sorted([c for c in set(dbscan_labels) if c != -1])
for c_id in cluster_ids:
    mask = (dbscan_labels == c_id)
    color = palette[c_id % len(palette)]
    ax.scatter(
        X[mask, 0], X[mask, 1],
        color=color, s=80, alpha=0.9,
        edgecolors='black', linewidth=0.8,
        label=f'Cluster {c_id} ({np.sum(mask)} customers)',
        zorder=3
    )

# 2. Plot noise points in bold black
noise_mask = (dbscan_labels == -1)
ax.scatter(
    X[noise_mask, 0], X[noise_mask, 1],
    color='#000000', marker='X', s=120, linewidth=1.5,
    label=f'Noise / Outliers ({n_noise} points)',
    zorder=4
)

ax.set_title("DBSCAN Clustering of Mall Customers (eps=0.35, min_samples=5)",
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Annual Income (k$)", fontsize=11, fontweight='bold')
ax.set_ylabel("Spending Score (1-100)", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(title="Cluster Identification", frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", loc="upper right")

plt.tight_layout()
plt.show()

## 3️⃣ Task 3: DBSCAN Hyperparameter Tuning (`eps` and `min_samples`)

### 📖 The Mechanics of Parameter Sensitivity
DBSCAN's performance depends heavily on the interplay between its two hyperparameters:
1. **`eps` (Neighborhood Radius):**
   - **Too Small:** Points cannot find enough neighbors to satisfy `min_samples`; clusters fragment into tiny pockets or get swallowed as noise.
   - **Too Large:** Distant density regions merge into a single undifferentiated macro-cluster, and true outliers get pulled into clusters.
2. **`min_samples` (Core Density Threshold):**
   - **Too Small (e.g., 2):** Weak density criteria; acts like single-linkage clustering, chaining noisy points together.
   - **Too Large:** Only exceptionally dense cores survive; moderate-density customer segments get discarded as noise.

We evaluate **4 distinct configurations** to observe these transitions systematically.

In [ ]:
# =====================================================================
# TASK 3: Parameter Sensitivity Analysis for DBSCAN
# =====================================================================
import pandas as pd
from sklearn.cluster import DBSCAN

parameter_grid = [
    {"eps": 0.25, "min_samples": 5, "setting": "Strict Radius (Small eps)"},
    {"eps": 0.35, "min_samples": 5, "setting": "Balanced Baseline"},
    {"eps": 0.45, "min_samples": 5, "setting": "Generous Radius (Large eps)"},
    {"eps": 0.35, "min_samples": 8, "setting": "Strict Density (High min_samples)"},
]

sensitivity_results = []
for config in parameter_grid:
    db = DBSCAN(eps=config["eps"], min_samples=config["min_samples"]).fit(X_scaled)
    labels = db.labels_
    n_clust = len(set(labels)) - (1 if -1 in labels else 0)
    n_nois = list(labels).count(-1)
    pct_noise = round((n_nois / len(X_scaled)) * 100, 1)
    
    sensitivity_results.append({
        "Configuration": config["setting"],
        "eps": config["eps"],
        "min_samples": config["min_samples"],
        "Clusters Found": n_clust,
        "Noise Count": n_nois,
        "Noise %": f"{pct_noise}%"
    })

df_sensitivity = pd.DataFrame(sensitivity_results)
print("=" * 80)
print("DBSCAN HYPERPARAMETER SENSITIVITY COMPARISON TABLE:")
print("=" * 80)
print(df_sensitivity.to_string(index=False))

### 📝 Parameter Effect Summary (3-4 Lines):
> **Key Finding:** Increasing `eps` expands the neighborhood radius, causing neighboring clusters to merge into fewer, larger groups while dramatically reducing the noise count (from 40+ points down to under 10). Conversely, increasing `min_samples` enforces a higher density threshold for core points, resulting in smaller, tighter clusters and converting borderline points into noise. Therefore, optimal clustering requires calibrating `eps` to bridge natural internal gaps within cohorts without accidentally bridging the valley between distinct customer segments.

## 4️⃣ Task 4: Comparative Study: DBSCAN vs. K-Means

### 📖 Theoretical & Practical Contrasts

| Dimension | 🔍 DBSCAN | 🎯 K-Means |
| :--- | :--- | :--- |
| **Cluster Geometry** | Can discover arbitrary, density-connected non-convex shapes. | Assumes spherical, isotropic convex clusters (Voronoi cells). |
| **Outlier & Noise Handling** | Identifies and rejects low-density outliers with label `-1`. | **Zero outlier immunity:** forces every distant outlier into a cluster, skewing centroids. |
| **Pre-specifying $k$** | **Not required:** Automatically discovers natural cluster count from density. | **Mandatory:** User must pre-define $k$ (e.g., via Elbow method). |
| **Computational Complexity** | $\mathcal{O}(N \log N)$ with spatial index trees; $\mathcal{O}(N^2)$ worst case. | $\mathcal{O}(N \cdot k \cdot d \cdot I)$, very fast on large datasets. |

---

### ⚖️ Experimental Setup:
- **Features:** `Annual Income (k$)` and `Spending Score (1-100)` (Standardized).
- **Constraint:** For fair comparison, we set K-Means' `n_clusters` equal to the number of non-noise clusters discovered by DBSCAN ($k=n_{\text{clusters}}$).

In [ ]:
# =====================================================================
# TASK 4: Side-by-Side Comparison of DBSCAN and K-Means
# =====================================================================
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# Fit K-Means using the same number of clusters found by DBSCAN
k_for_kmeans = n_clusters
kmeans = KMeans(n_clusters=k_for_kmeans, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)

# Visualization: Side-by-Side Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Left Subplot: DBSCAN
for c_id in cluster_ids:
    mask = (dbscan_labels == c_id)
    ax1.scatter(X[mask, 0], X[mask, 1], s=70, edgecolors='black', linewidth=0.6,
                label=f'Cluster {c_id}', alpha=0.9)

ax1.scatter(X[noise_mask, 0], X[noise_mask, 1], color='black', marker='X', s=110,
            label='Noise (-1)', linewidth=1.5, zorder=5)
ax1.set_title(f"DBSCAN (Discovered k={n_clusters}, Noise={n_noise})", fontsize=12, fontweight='bold')
ax1.set_xlabel("Annual Income (k$)", fontsize=11, fontweight='bold')
ax1.set_ylabel("Spending Score (1-100)", fontsize=11, fontweight='bold')
ax1.grid(True, linestyle='--', alpha=0.5)
ax1.legend(loc="upper right", fontsize=9)

# Right Subplot: K-Means
for k_id in range(k_for_kmeans):
    mask = (kmeans_labels == k_id)
    ax2.scatter(X[mask, 0], X[mask, 1], s=70, edgecolors='black', linewidth=0.6,
                label=f'Cluster {k_id}', alpha=0.9)

# Plot K-Means centroids
centroids_orig = scaler.inverse_transform(kmeans.cluster_centers_)
ax2.scatter(centroids_orig[:, 0], centroids_orig[:, 1], marker='*', s=260, color='red',
            edgecolors='black', linewidth=1.2, label='Centroids', zorder=6)

ax2.set_title(f"K-Means (Forced k={k_for_kmeans}, Zero Noise Isolation)", fontsize=12, fontweight='bold')
ax2.set_xlabel("Annual Income (k$)", fontsize=11, fontweight='bold')
ax2.set_ylabel("Spending Score (1-100)", fontsize=11, fontweight='bold')
ax2.grid(True, linestyle='--', alpha=0.5)
ax2.legend(loc="upper right", fontsize=9)

plt.tight_layout()
plt.show()

### 🔍 Comparative Observations: Which Method Better Handles Outliers & Shapes?
1. **Outlier Handling:** **DBSCAN is vastly superior** at handling outliers. In the left plot, genuine customer outliers (e.g., extreme high-income individuals with atypical spending) are flagged as noise (`-1`) without distorting the core clusters. In contrast, K-Means (right plot) is forced to assign every outlier to its nearest centroid, pulling the centroids away from true density centers.
2. **Cluster Shape Adaptability:** K-Means enforces spherical boundaries equidistant from its centroids. If clusters are elongated, crescent-shaped, or unequal in density, K-Means fractures them unnaturally. DBSCAN follows dense contours regardless of geometric shape.

## 5️⃣ Task 5: Clustering Non-Spherical Data (`make_moons`) & Noise Extraction

### 📖 The Challenge of Non-Convex Manifolds
- Standard centroid-based algorithms like K-Means calculate distance from an imaginary center point $\mu$. When data forms interlocking crescents (like `make_moons`), the geometric center of each crescent actually lies in empty space between the moons.
- **DBSCAN's Density Reachability:** DBSCAN walks from point to point within radius $\epsilon$. As long as points are connected by high density, DBSCAN traces the entire curved crescent effortlessly.
- In this task, we generate a two-moon dataset with added Gaussian noise, cluster it with DBSCAN, extract the exact indices of all noise points (`label == -1`), and print them.

In [ ]:
# =====================================================================
# TASK 5: Clustering Non-Spherical Manifold (make_moons) with DBSCAN
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN

# 1. Generate non-spherical synthetic moon dataset with noise
X_moons, y_true = make_moons(n_samples=250, noise=0.10, random_state=42)

# Add 10 explicit random background outlier points to make noise identification prominent
np.random.seed(42)
outliers = np.random.uniform(low=[-1.5, -1.0], high=[2.5, 1.5], size=(10, 2))
X_moons_with_outliers = np.vstack([X_moons, outliers])

# 2. Apply DBSCAN (calibrated for the moon manifold)
db_moons = DBSCAN(eps=0.20, min_samples=6)
moon_labels = db_moons.fit_predict(X_moons_with_outliers)

# 3. Extract and print indices of all noise points (label == -1)
noise_indices = np.where(moon_labels == -1)[0]
n_moons_clusters = len(set(moon_labels)) - (1 if -1 in moon_labels else 0)

print("=" * 75)
print("TASK 5: NON-SPHERICAL MOON CLUSTERING & NOISE DETECTION RESULTS")
print("=" * 75)
print(f"Total Points Evaluated          : {len(X_moons_with_outliers)}")
print(f"Dense Clusters Identified       : {n_moons_clusters} (The two crescent moons)")
print(f"Total Noise Points Detected     : {len(noise_indices)}")
print(f"\nDetected Noise Point Indices   :\n{noise_indices.tolist()}")
print("\nCoordinates of First 5 Noise Outliers:")
for idx in noise_indices[:5]:
    print(f"  Index {idx:03d} -> Coordinates: {X_moons_with_outliers[idx].round(3).tolist()}")

# 4. Visualization of the Clustered Moon Manifold
fig, ax = plt.subplots(figsize=(10, 6))
moon_colors = {0: '#3498db', 1: '#e67e22'}  # Blue for Moon 1, Orange for Moon 2

for cluster_id in range(n_moons_clusters):
    mask = (moon_labels == cluster_id)
    ax.scatter(X_moons_with_outliers[mask, 0], X_moons_with_outliers[mask, 1],
               color=moon_colors.get(cluster_id, 'green'), s=75, alpha=0.9,
               edgecolors='black', linewidth=0.6, label=f'Moon Cluster {cluster_id}')

# Highlight noise points in black
ax.scatter(X_moons_with_outliers[noise_indices, 0], X_moons_with_outliers[noise_indices, 1],
           color='#000000', marker='X', s=130, linewidth=1.5,
           label=f'Noise / Outliers ({len(noise_indices)} points)', zorder=5)

ax.set_title("DBSCAN on Non-Spherical 'make_moons' Dataset with Isolated Noise",
             fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel("Feature Dimension 1 (X1)", fontsize=11, fontweight='bold')
ax.set_ylabel("Feature Dimension 2 (X2)", fontsize=11, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5)
ax.legend(frameon=True, facecolor="#fdfefe", edgecolor="#cccccc", loc="upper right")

plt.tight_layout()
plt.show()

## 🏁 Session 3 Summary & Takeaways

1. **Density vs. Centroid Paradigm:** DBSCAN shifts the clustering philosophy from centroid proximity to local point density connectivity, eliminating the rigid spherical cluster constraint.
2. **Native Outlier Rejection:** Unlike K-Means, which forcibly assigns every data point to a cluster, DBSCAN natively flags anomalous, low-density points with label `-1`, protecting cluster definitions from outlier corruption.
3. **Parameter Balancing (`eps`, `min_samples`):** Parameter choice governs the clustering resolution. Small `eps` or large `min_samples` creates high-purity, conservative clusters with high noise rejection; large `eps` or small `min_samples` bridges natural clusters and reduces noise.
4. **Manifold Versatility:** DBSCAN excels on complex, non-linear manifolds (`make_moons`, `make_circles`) where distance-to-centroid methods completely break down.